# CORE MODELS

Core model pipeline using `LASSO_CORE20`, `LGBM_CORE20`, `NN3_CORE20`, `TR_CORE20`, and `TR_LAG_CLOUD`.

`run_pipeline(args)` executes the selected models. Existing pooled OOS prediction files are loaded and used to refresh metrics/backtests; missing prediction files trigger model training.


## Colab setup


In [1]:
try:
    import google.colab  # type: ignore
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

import os, sys, subprocess
print('IN_COLAB:', IN_COLAB)
try:
    subprocess.run(['nvidia-smi'], check=False)
except Exception as e:
    print('nvidia-smi unavailable:', e)


Mounted at /content/drive
IN_COLAB: True


## Pipeline definitions

In [2]:
"""
Core20-only modelling pipeline for the Quant Research Project.

Implements:
    LASSO_CORE20
    LGBM_CORE20
    NN3_CORE20
    TR_CORE20
    TR_LAG_CLOUD

The raw input set is restricted to the Core20 characteristics.
The `TR_LAG_CLOUD` feature set contains dynamic cloud characteristics constructed from Core20.

Core safeguards:
    - 15-year rolling training, 4-year validation, 1-year test, annual refits.
    - No dropping missing future returns before rank-normalization.
    - USA models use month-level rank normalization.
    - Dynamic cloud features use only t and t-1 information.
    - Transformer attention clouds include all stocks available at month t after current-date filters, including stocks with missing t+1 target.
    - Loss/predictions use only rows with observed next-month returns.
    - No positional embeddings based on arbitrary stock order.
    - Checkpoint/resume logic with config hashes. Reruns skip completed refits.
"""

from __future__ import annotations

import argparse
import gc
import hashlib
import json
import math
import pickle
import random
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset, TensorDataset
except ImportError as exc:
    raise ImportError("This pipeline requires PyTorch.") from exc

try:
    from sklearn.linear_model import Lasso
    from sklearn.metrics import mean_squared_error
except ImportError as exc:
    raise ImportError("This pipeline requires scikit-learn. In Colab: pip install scikit-learn") from exc

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm", "-q"])
        import lightgbm as lgb
        from lightgbm import LGBMRegressor
    except Exception:
        lgb = None
        LGBMRegressor = None


## 1. Core20 characteristics


In [3]:
# =============================================================================
# 1. Core20 characteristics only
# =============================================================================

# Raw firm-characteristic input set for the Core20 models.
# TR_LAG_CLOUD constructs its 100 dynamic cloud features from these Core20 variables only.
CORE20: List[str] = [
    "market_equity", "be_me", "ret_12_1", "ret_1_0", "ret_6_1",
    "ret_60_12", "ope_be", "gp_at", "at_gr1", "inv_gr1",
    "netis_at", "oaccruals_at", "ivol_capm_21d", "beta_60m", "rmax5_21d",
    "turnover_126d", "dolvol_126d", "ami_126d", "debt_me", "niq_at",
]

RAW_FEATURE_SETS = {20: CORE20}
REQUIRED_RAW_FEATURES = CORE20

assert len(CORE20) == 20
assert len(set(CORE20)) == 20


## 2. Configs

In [4]:
# =============================================================================
# 2. Configs
# =============================================================================

@dataclass(frozen=True)
class WindowConfig:
    train_years: int = 15
    val_years: int = 4
    test_years: int = 1
    first_year: int = 1980
    last_year: int = 2024


@dataclass(frozen=True)
class TrainConfig:
    seed: int = 42
    batch_size_transformer: int = 2
    batch_size_nn3: int = 8192
    max_epochs_transformer: int = 80
    max_epochs_nn3: int = 80
    patience: int = 10
    lr_transformer: float = 1e-3
    lr_nn3: float = 1e-3
    weight_decay: float = 1e-4
    l1_lambda_nn3: float = 1e-5
    num_workers: int = 0


@dataclass(frozen=True)
class TransformerConfig:
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 2
    d_ff: int = 128
    dropout: float = 0.10
    country_embed_dim: int = 4


@dataclass(frozen=True)
class LassoConfig:
    alphas: Tuple[float, ...] = (1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2)
    max_iter: int = 5000


@dataclass(frozen=True)
class LGBMConfig:
    n_estimators: int = 3000
    learning_rate: float = 0.03
    num_leaves: int = 31
    max_depth: int = -1
    min_child_samples: int = 200
    subsample: float = 0.80
    colsample_bytree: float = 0.80
    reg_alpha: float = 0.0
    reg_lambda: float = 1.0
    early_stopping_rounds: int = 100
    n_jobs: int = -1
    seed: int = 42


@dataclass(frozen=True)
class ModelSpec:
    model_id: str
    family: str                    # "lasso", "lgbm", "nn3", "transformer"
    universe: str                  # "USA" or "DEV"
    feature_mode: str              # "raw" or "lag_cloud"
    raw_count: int                 # 20/40/60/80/100; 20 for lag_cloud base
    use_country_embedding: bool = False


MODEL_SPECS: Dict[str, ModelSpec] = {
    # Core20 raw-feature benchmarks
    "LASSO_CORE20": ModelSpec("LASSO_CORE20", "lasso", "USA", "raw", 20),
    "LGBM_CORE20": ModelSpec("LGBM_CORE20", "lgbm", "USA", "raw", 20),
    "NN3_CORE20": ModelSpec("NN3_CORE20", "nn3", "USA", "raw", 20),
    "TR_CORE20": ModelSpec("TR_CORE20", "transformer", "USA", "raw", 20),

    # Proposed dynamic cloud model: 100 engineered features constructed from Core20 only
    "TR_LAG_CLOUD": ModelSpec("TR_LAG_CLOUD", "transformer", "USA", "lag_cloud", 20),
}

DEFAULT_MODELS = list(MODEL_SPECS.keys())

## 3. Utilities

In [5]:
# =============================================================================
# 3. Utilities
# =============================================================================

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def safe_to_float64(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").astype("float64")


def json_hash(obj: dict) -> str:
    text = json.dumps(obj, sort_keys=True, default=str)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def save_json(obj: dict, path: Path) -> None:
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)


def load_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def oos_r2(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.sum(y_true ** 2)
    if denom <= 0 or not np.isfinite(denom):
        return np.nan
    return float(1.0 - np.sum((y_true - y_pred) ** 2) / denom)


def year_mask(df_or_dates, start_year: int, end_year: int) -> np.ndarray:
    if isinstance(df_or_dates, pd.DataFrame):
        years = df_or_dates["eom"].dt.year.to_numpy()
    else:
        years = pd.to_datetime(df_or_dates).year.to_numpy()
    return (years >= start_year) & (years <= end_year)

## 4. Data loading and no-lookahead preprocessing

In [6]:
# =============================================================================
# 4. Data loading and no-lookahead preprocessing
# =============================================================================

def load_panel(data_path: Path, start_year: int, end_year: int) -> pd.DataFrame:
    print(f"Loading data from: {data_path}")
    if not data_path.exists():
        raise FileNotFoundError(data_path)
    df = pd.read_parquet(data_path)
    df["eom"] = pd.to_datetime(df["eom"])
    if "excntry" not in df.columns:
        raise ValueError("Input panel must contain 'excntry' for country filtering.")
    df = df[(df["eom"].dt.year >= start_year) & (df["eom"].dt.year <= end_year)].copy()
    print(f"Raw filtered shape: {df.shape}")
    print(f"Date range: {df['eom'].min()} to {df['eom'].max()}")
    print("Countries in file:", sorted(df["excntry"].dropna().unique())[:30])
    return df


def validate_columns(df: pd.DataFrame, required_features: List[str], target_col: str) -> None:
    base_required = ["id", "eom", "excntry", target_col]
    missing = [c for c in base_required + required_features if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def apply_country_month_size_cap(df: pd.DataFrame, cap: Optional[int], me_col: str = "me") -> pd.DataFrame:
    """Uses month-t market equity only. This is not lookahead."""
    if cap is None or cap <= 0:
        return df
    if me_col not in df.columns:
        print("WARNING: me column missing; cannot apply country-month size cap.")
        return df
    out = df.copy()
    out[me_col] = pd.to_numeric(out[me_col], errors="coerce")
    out["_rank_me_desc"] = out.groupby(["excntry", "eom"])[me_col].rank(method="first", ascending=False, na_option="bottom")
    out = out[out["_rank_me_desc"] <= cap].drop(columns=["_rank_me_desc"]).copy()
    print(f"Applied country-month size cap {cap}: {len(df):,} -> {len(out):,} rows")
    return out


def rank_normalize_no_lookahead(
    df: pd.DataFrame,
    feature_cols: List[str],
    target_col: str,
    universe: str,
    dev_size_cap: Optional[int] = None,
) -> pd.DataFrame:
    """
    No-lookahead preprocessing. Does NOT drop missing future returns before ranks.

    USA: ranks within eom.
    """
    work = df.copy()
    if universe != "USA":
        raise ValueError("This pipeline is USA/Core20-only; universe must be 'USA'.")
    work = work[work["excntry"].eq("USA")].copy()

    keep_cols = ["id", "eom", "excntry", target_col, "me", "permno", "gvkey", "ff49", "size_grp", "ret_exc"]
    keep_cols = [c for c in keep_cols if c in work.columns]
    keep_cols = list(dict.fromkeys(keep_cols + feature_cols))
    work = work[keep_cols].dropna(subset=["id", "eom", "excntry"]).copy()
    work["eom"] = pd.to_datetime(work["eom"])

    for c in feature_cols + [target_col, "me"]:
        if c in work.columns:
            work[c] = safe_to_float64(work[c])

    group_cols = ["eom"]
    print(f"Rank-normalizing {universe} by {group_cols} BEFORE dropping missing future returns...")
    g = work.groupby(group_cols, sort=False)[feature_cols]
    ranks = g.rank(method="average", na_option="keep")
    counts = g.transform("count")
    denom = (counts - 1).replace(0, np.nan)
    normed = 2.0 * (ranks - 1.0) / denom - 1.0
    work.loc[:, feature_cols] = normed.astype("float32")
    work[feature_cols] = work[feature_cols].fillna(0.0).astype("float32")

    work = work.sort_values(["eom", "excntry", "id"]).reset_index(drop=True)
    work["country_id"] = 0
    print(f"{universe} normalized shape: {work.shape}")
    return work


def add_lag_cloud_features(work: pd.DataFrame, core_features: List[str], universe: str, max_gap_days: int = 45) -> Tuple[pd.DataFrame, List[str]]:
    """
    z_i,t = [x_i,t, x_i,t-1, x_i,t - x_i,t-1, x_i,t - c_t, x_i,t - c_t-1]

    USA centroids are month-level.
    """
    df = work.copy()
    df["eom"] = pd.to_datetime(df["eom"])
    df = df.sort_values(["excntry", "id", "eom"]).reset_index(drop=True)

    lag_cols, vel_cols = [], []
    g = df.groupby(["excntry", "id"], sort=False)
    prev_date = g["eom"].shift(1)
    gap_days = (df["eom"] - prev_date).dt.days
    valid_lag = gap_days.le(max_gap_days) & gap_days.ge(20)

    for c in core_features:
        lag_col = f"{c}__lag1"
        vel_col = f"{c}__vel1"
        raw_lag = g[c].shift(1)
        df[lag_col] = raw_lag.where(valid_lag, 0.0).fillna(0.0).astype("float32")
        df[vel_col] = (df[c] - raw_lag).where(valid_lag, 0.0).fillna(0.0).astype("float32")
        lag_cols.append(lag_col)
        vel_cols.append(vel_col)

    centroid_groups = ["eom"]
    cent = df.groupby(centroid_groups, sort=False)[core_features].mean().reset_index()
    cent = cent.rename(columns={c: f"{c}__centroid_t" for c in core_features})
    df = df.merge(cent, on=centroid_groups, how="left", validate="many_to_one")

    rel_current_cols = []
    for c in core_features:
        rel_col = f"{c}__rel_centroid_t"
        df[rel_col] = (df[c] - df[f"{c}__centroid_t"]).fillna(0.0).astype("float32")
        rel_current_cols.append(rel_col)

    cent_prev = cent.copy()
    cent_prev["eom"] = cent_prev["eom"] + pd.offsets.MonthEnd(1)
    cent_prev = cent_prev.rename(columns={f"{c}__centroid_t": f"{c}__centroid_tminus1" for c in core_features})
    df = df.merge(cent_prev, on=centroid_groups, how="left", validate="many_to_one")

    rel_prior_cols = []
    prev_cent_cols = [f"{c}__centroid_tminus1" for c in core_features]
    has_prev_centroid = df[prev_cent_cols].notna().all(axis=1)
    for c in core_features:
        rel_col = f"{c}__rel_centroid_tminus1"
        raw_rel = df[c] - df[f"{c}__centroid_tminus1"]
        df[rel_col] = raw_rel.where(has_prev_centroid, 0.0).fillna(0.0).astype("float32")
        rel_prior_cols.append(rel_col)

    drop_centroid_cols = [f"{c}__centroid_t" for c in core_features] + prev_cent_cols
    df = df.drop(columns=drop_centroid_cols)

    lag_cloud_features = core_features + lag_cols + vel_cols + rel_current_cols + rel_prior_cols
    assert len(lag_cloud_features) == 100
    assert len(set(lag_cloud_features)) == 100
    return df, lag_cloud_features


def build_feature_frame_for_model(normalized_work: pd.DataFrame, spec: ModelSpec, target_col: str) -> Tuple[pd.DataFrame, List[str]]:
    if spec.feature_mode == "raw":
        features = RAW_FEATURE_SETS[spec.raw_count]
        cols = ["id", "eom", "excntry", "country_id", target_col] + [c for c in ["me", "permno", "gvkey", "ff49", "size_grp"] if c in normalized_work.columns] + features
        return normalized_work[cols].copy(), features
    if spec.feature_mode == "lag_cloud":
        out, features = add_lag_cloud_features(normalized_work, CORE20, spec.universe)
        cols = ["id", "eom", "excntry", "country_id", target_col] + [c for c in ["me", "permno", "gvkey", "ff49", "size_grp"] if c in out.columns] + features
        return out[cols].copy(), features
    raise ValueError(spec.feature_mode)

## 5. Rolling schedule

In [7]:
# =============================================================================
# 5. Rolling schedule
# =============================================================================

def make_rolling_schedule(work: pd.DataFrame, window_cfg: WindowConfig) -> pd.DataFrame:
    min_year = int(max(work["eom"].dt.year.min(), window_cfg.first_year))
    max_year = int(min(work["eom"].dt.year.max(), window_cfg.last_year))
    first_test_year = min_year + window_cfg.train_years + window_cfg.val_years
    rows = []
    for refit_id, test_year in enumerate(range(first_test_year, max_year + 1)):
        rows.append({
            "refit_id": refit_id,
            "test_year": test_year,
            "train_start_year": test_year - window_cfg.val_years - window_cfg.train_years,
            "train_end_year": test_year - window_cfg.val_years - 1,
            "val_start_year": test_year - window_cfg.val_years,
            "val_end_year": test_year - 1,
            "test_start_year": test_year,
            "test_end_year": test_year,
        })
    sched = pd.DataFrame(rows)
    if sched.empty:
        raise ValueError("No rolling windows available. Check sample range and window lengths.")
    return sched

## 6. Datasets and models

In [8]:
# =============================================================================
# 6. Datasets and models
# =============================================================================

class MonthPanelDataset(Dataset):
    def __init__(self, X, Y, EXISTS, LABEL, COUNTRY, month_indices):
        self.X = X[month_indices]
        self.Y = Y[month_indices]
        self.EXISTS = EXISTS[month_indices]
        self.LABEL = LABEL[month_indices]
        self.COUNTRY = COUNTRY[month_indices]
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.X[idx]),
            torch.from_numpy(self.Y[idx]),
            torch.from_numpy(self.EXISTS[idx]),
            torch.from_numpy(self.LABEL[idx]),
            torch.from_numpy(self.COUNTRY[idx]),
        )


class CrossSectionalTransformer(nn.Module):
    """Set-style Transformer over stocks in a month. No positional embeddings."""
    def __init__(self, n_features: int, cfg: TransformerConfig, use_country_embedding: bool = False, num_countries: int = 1):
        super().__init__()
        self.use_country_embedding = use_country_embedding
        self.country_embedding = None
        input_dim = n_features
        if use_country_embedding:
            self.country_embedding = nn.Embedding(num_countries, cfg.country_embed_dim)
            input_dim += cfg.country_embed_dim
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, cfg.d_model),
            nn.LayerNorm(cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
        )
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=cfg.d_ff,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
        self.head = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Linear(cfg.d_model, 1))

    def forward(self, x: torch.Tensor, exists_mask: torch.Tensor, country_ids: Optional[torch.Tensor] = None) -> torch.Tensor:
        if self.use_country_embedding:
            if country_ids is None:
                raise ValueError("country_ids required when use_country_embedding=True")
            emb = self.country_embedding(country_ids.long())
            x = torch.cat([x, emb], dim=-1)
        h = self.input_proj(x)
        key_padding_mask = ~exists_mask.bool()
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)
        return self.head(h).squeeze(-1)


class GKXStyleNN3(nn.Module):
    def __init__(self, n_features: int, dropout: float = 0.05):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 16), nn.BatchNorm1d(16), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(16, 8), nn.BatchNorm1d(8), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(8, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


def l1_linear_weights(model: nn.Module) -> torch.Tensor:
    penalty = torch.tensor(0.0, device=next(model.parameters()).device)
    for module in model.modules():
        if isinstance(module, nn.Linear):
            penalty = penalty + module.weight.abs().sum()
    return penalty

## 7. Panelization and flat arrays

In [9]:
# =============================================================================
# 7. Panelization and flat arrays
# =============================================================================

def panelize_for_transformer(df_features: pd.DataFrame, feature_cols: List[str], target_col: str) -> dict:
    """Includes all available month-t stocks in attention cloud; LABEL marks observed target rows."""
    df = df_features.copy()
    df["eom"] = pd.to_datetime(df["eom"])
    df = df.sort_values(["eom", "excntry", "id"]).reset_index(drop=True)
    months = pd.Index(sorted(df["eom"].unique()))
    month_groups = df.groupby("eom", sort=False)
    n_max = int(month_groups.size().max())
    t_len = len(months)
    k = len(feature_cols)

    X = np.zeros((t_len, n_max, k), dtype=np.float32)
    Y = np.zeros((t_len, n_max), dtype=np.float32)
    EXISTS = np.zeros((t_len, n_max), dtype=bool)
    LABEL = np.zeros((t_len, n_max), dtype=bool)
    COUNTRY_PANEL = np.zeros((t_len, n_max), dtype=np.int64)
    ID_PANEL = np.full((t_len, n_max), "", dtype=object)
    COUNTRY_STR_PANEL = np.full((t_len, n_max), "", dtype=object)
    ME_PANEL = np.full((t_len, n_max), np.nan, dtype=np.float64)

    for t, month in enumerate(months):
        sub = month_groups.get_group(month).sort_values(["excntry", "id"]).reset_index(drop=True)
        n = len(sub)
        X[t, :n, :] = sub[feature_cols].to_numpy(dtype=np.float32, copy=True)
        y = sub[target_col].to_numpy(dtype=np.float64, copy=True)
        valid_y = np.isfinite(y)
        Y[t, :n] = np.nan_to_num(y, nan=0.0).astype(np.float32)
        EXISTS[t, :n] = True
        LABEL[t, :n] = valid_y
        COUNTRY_PANEL[t, :n] = sub["country_id"].to_numpy(dtype=np.int64, copy=True) if "country_id" in sub.columns else 0
        ID_PANEL[t, :n] = sub["id"].astype(str).to_numpy()
        COUNTRY_STR_PANEL[t, :n] = sub["excntry"].astype(str).to_numpy()
        if "me" in sub.columns:
            ME_PANEL[t, :n] = pd.to_numeric(sub["me"], errors="coerce").to_numpy(dtype=np.float64)

    return {"months": months, "X": X, "Y": Y, "EXISTS": EXISTS, "LABEL": LABEL, "COUNTRY": COUNTRY_PANEL, "ID_PANEL": ID_PANEL, "COUNTRY_STR_PANEL": COUNTRY_STR_PANEL, "ME_PANEL": ME_PANEL}


def flat_labeled_arrays(df_features: pd.DataFrame, feature_cols: List[str], target_col: str) -> pd.DataFrame:
    df = df_features.copy()
    df["eom"] = pd.to_datetime(df["eom"])
    df = df.dropna(subset=[target_col]).copy()
    df[feature_cols] = df[feature_cols].astype("float32")
    return df.sort_values(["eom", "excntry", "id"]).reset_index(drop=True)

## 8. Checkpoints

In [10]:
# =============================================================================
# 8. Checkpoints
# =============================================================================

def checkpoint_paths(base_dir: Path, model_id: str, refit_id: int) -> dict:
    refit_dir = base_dir / model_id / f"refit_{refit_id:03d}"
    ensure_dir(refit_dir)
    return {
        "dir": refit_dir,
        "metadata": refit_dir / "metadata.json",
        "latest": refit_dir / "latest.pt",
        "best": refit_dir / "best.pt",
        "lasso_model": refit_dir / "lasso_model.pkl",
        "lgbm_model": refit_dir / "lgbm_model.pkl",
        "preds": refit_dir / "test_predictions.parquet",
    }


def metadata_matches(path: Path, config_hash: str) -> bool:
    if not path.exists():
        return False
    try:
        return load_json(path).get("config_hash") == config_hash
    except Exception:
        return False

## 9. Training and prediction routines

In [11]:
# =============================================================================
# 9. Training and prediction routines
# =============================================================================

def train_lasso_refit(model_id, flat_df, feature_cols, sched_row, checkpoint_base, lasso_cfg, config_hash, target_col):
    paths = checkpoint_paths(checkpoint_base, model_id, int(sched_row.refit_id))
    if paths["preds"].exists() and metadata_matches(paths["metadata"], config_hash):
        print(f"  {model_id} refit {sched_row.refit_id}: predictions exist; loading.")
        return pd.read_parquet(paths["preds"])

    train = flat_df[year_mask(flat_df, sched_row.train_start_year, sched_row.train_end_year)].copy()
    val = flat_df[year_mask(flat_df, sched_row.val_start_year, sched_row.val_end_year)].copy()
    test = flat_df[year_mask(flat_df, sched_row.test_start_year, sched_row.test_end_year)].copy()
    if min(len(train), len(val), len(test)) == 0:
        raise ValueError(f"Empty split for {model_id} refit {sched_row.refit_id}")

    X_train = train[feature_cols].to_numpy(dtype=np.float32)
    y_train = train[target_col].to_numpy(dtype=np.float64)
    X_val = val[feature_cols].to_numpy(dtype=np.float32)
    y_val = val[target_col].to_numpy(dtype=np.float64)
    X_test = test[feature_cols].to_numpy(dtype=np.float32)

    best_alpha, best_val, best_model = None, float("inf"), None
    for alpha in lasso_cfg.alphas:
        model = Lasso(alpha=alpha, fit_intercept=True, max_iter=lasso_cfg.max_iter, random_state=42)
        model.fit(X_train, y_train)
        pred_val = model.predict(X_val)
        val_mse = mean_squared_error(y_val, pred_val)
        if val_mse < best_val:
            best_alpha, best_val, best_model = alpha, val_mse, model
    with open(paths["lasso_model"], "wb") as f:
        pickle.dump(best_model, f)

    save_json({"config_hash": config_hash, "model_id": model_id, "refit_id": int(sched_row.refit_id), "feature_cols": feature_cols, "schedule": sched_row.to_dict(), "best_alpha": best_alpha, "best_val_mse": best_val, "lasso_cfg": asdict(lasso_cfg)}, paths["metadata"])
    y_pred = best_model.predict(X_test)
    preds = pd.DataFrame({
        "eom": test["eom"].to_numpy(),
        "id": test["id"].astype(str).to_numpy(),
        "country": test["excntry"].astype(str).to_numpy(),
        "me": pd.to_numeric(test["me"], errors="coerce").to_numpy(dtype=np.float64) if "me" in test.columns else np.nan,
        "y_true": test[target_col].to_numpy(dtype=np.float64),
        "y_pred": y_pred.astype(np.float64),
        "model_id": model_id,
        "refit_id": int(sched_row.refit_id),
        "test_year": int(sched_row.test_year),
    })
    preds.to_parquet(paths["preds"], index=False)
    print(f"  {model_id} refit {sched_row.refit_id}: alpha={best_alpha}, saved predictions {preds.shape}, R2={oos_r2(preds.y_true, preds.y_pred):.4%}")
    return preds


def train_lgbm_refit(model_id, flat_df, feature_cols, sched_row, checkpoint_base, lgbm_cfg, config_hash, target_col):
    paths = checkpoint_paths(checkpoint_base, model_id, int(sched_row.refit_id))
    if paths["preds"].exists() and metadata_matches(paths["metadata"], config_hash):
        print(f"  {model_id} refit {sched_row.refit_id}: predictions exist; loading.")
        return pd.read_parquet(paths["preds"])

    if lgb is None or LGBMRegressor is None:
        raise ImportError("LightGBM is required for LGBM models. In Colab, run: !pip install lightgbm")

    train = flat_df[year_mask(flat_df, sched_row.train_start_year, sched_row.train_end_year)].copy()
    val = flat_df[year_mask(flat_df, sched_row.val_start_year, sched_row.val_end_year)].copy()
    test = flat_df[year_mask(flat_df, sched_row.test_start_year, sched_row.test_end_year)].copy()
    if min(len(train), len(val), len(test)) == 0:
        raise ValueError(f"Empty split for {model_id} refit {sched_row.refit_id}")

    X_train = train[feature_cols].to_numpy(dtype=np.float32)
    y_train = train[target_col].to_numpy(dtype=np.float64)
    X_val = val[feature_cols].to_numpy(dtype=np.float32)
    y_val = val[target_col].to_numpy(dtype=np.float64)
    X_test = test[feature_cols].to_numpy(dtype=np.float32)

    model = LGBMRegressor(
        objective="regression",
        boosting_type="gbdt",
        n_estimators=lgbm_cfg.n_estimators,
        learning_rate=lgbm_cfg.learning_rate,
        num_leaves=lgbm_cfg.num_leaves,
        max_depth=lgbm_cfg.max_depth,
        min_child_samples=lgbm_cfg.min_child_samples,
        subsample=lgbm_cfg.subsample,
        subsample_freq=1,
        colsample_bytree=lgbm_cfg.colsample_bytree,
        reg_alpha=lgbm_cfg.reg_alpha,
        reg_lambda=lgbm_cfg.reg_lambda,
        random_state=lgbm_cfg.seed,
        n_jobs=lgbm_cfg.n_jobs,
        verbosity=-1,
    )
    callbacks = [
        lgb.early_stopping(stopping_rounds=lgbm_cfg.early_stopping_rounds, verbose=False),
        lgb.log_evaluation(period=0),
    ]
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="l2", callbacks=callbacks)
    pred_val = model.predict(X_val, num_iteration=model.best_iteration_)
    val_mse = mean_squared_error(y_val, pred_val)

    with open(paths["lgbm_model"], "wb") as f:
        pickle.dump(model, f)

    save_json({
        "config_hash": config_hash,
        "model_id": model_id,
        "refit_id": int(sched_row.refit_id),
        "feature_cols": feature_cols,
        "schedule": sched_row.to_dict(),
        "best_iteration": int(model.best_iteration_ or lgbm_cfg.n_estimators),
        "best_val_mse": float(val_mse),
        "lgbm_cfg": asdict(lgbm_cfg),
    }, paths["metadata"])

    y_pred = model.predict(X_test, num_iteration=model.best_iteration_)
    preds = pd.DataFrame({
        "eom": test["eom"].to_numpy(),
        "id": test["id"].astype(str).to_numpy(),
        "country": test["excntry"].astype(str).to_numpy(),
        "me": pd.to_numeric(test["me"], errors="coerce").to_numpy(dtype=np.float64) if "me" in test.columns else np.nan,
        "y_true": test[target_col].to_numpy(dtype=np.float64),
        "y_pred": y_pred.astype(np.float64),
        "model_id": model_id,
        "refit_id": int(sched_row.refit_id),
        "test_year": int(sched_row.test_year),
    })
    preds.to_parquet(paths["preds"], index=False)
    print(
        f"  {model_id} refit {sched_row.refit_id}: "
        f"best_iter={int(model.best_iteration_ or lgbm_cfg.n_estimators)}, "
        f"val_mse={val_mse:.6g}, saved predictions {preds.shape}, "
        f"R2={oos_r2(preds.y_true, preds.y_pred):.4%}"
    )
    return preds


def train_transformer_refit(model_id, spec, panel, feature_cols, sched_row, checkpoint_base, train_cfg, tr_cfg, device, config_hash):
    paths = checkpoint_paths(checkpoint_base, model_id, int(sched_row.refit_id))
    if paths["preds"].exists() and metadata_matches(paths["metadata"], config_hash):
        print(f"  {model_id} refit {sched_row.refit_id}: predictions exist; loading.")
        return pd.read_parquet(paths["preds"])

    months = panel["months"]
    train_idx = np.where(year_mask(months, sched_row.train_start_year, sched_row.train_end_year))[0]
    val_idx = np.where(year_mask(months, sched_row.val_start_year, sched_row.val_end_year))[0]
    test_idx = np.where(year_mask(months, sched_row.test_start_year, sched_row.test_end_year))[0]
    if min(len(train_idx), len(val_idx), len(test_idx)) == 0:
        raise ValueError(f"Empty split for {model_id} refit {sched_row.refit_id}")

    train_ds = MonthPanelDataset(panel["X"], panel["Y"], panel["EXISTS"], panel["LABEL"], panel["COUNTRY"], train_idx)
    val_ds = MonthPanelDataset(panel["X"], panel["Y"], panel["EXISTS"], panel["LABEL"], panel["COUNTRY"], val_idx)
    train_loader = DataLoader(train_ds, batch_size=train_cfg.batch_size_transformer, shuffle=True, num_workers=train_cfg.num_workers)
    val_loader = DataLoader(val_ds, batch_size=train_cfg.batch_size_transformer, shuffle=False, num_workers=train_cfg.num_workers)

    num_countries = 1  # USA-only pipeline
    model = CrossSectionalTransformer(len(feature_cols), tr_cfg, spec.use_country_embedding, num_countries).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr_transformer, weight_decay=train_cfg.weight_decay)

    start_epoch, best_val, bad_epochs = 0, float("inf"), 0
    if paths["latest"].exists() and metadata_matches(paths["metadata"], config_hash):
        ckpt = torch.load(paths["latest"], map_location=device)
        model.load_state_dict(ckpt["model_state"])
        opt.load_state_dict(ckpt["optimizer_state"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_val = float(ckpt.get("best_val", best_val))
        bad_epochs = int(ckpt.get("bad_epochs", bad_epochs))
        print(f"  {model_id} refit {sched_row.refit_id}: resumed from epoch {start_epoch}.")

    def run_epoch(loader, train_mode: bool) -> float:
        model.train(train_mode)
        total_loss, total_n = 0.0, 0
        for xb, yb, exists, label, country in loader:
            xb, yb, exists, label, country = xb.to(device), yb.to(device), exists.to(device), label.to(device), country.to(device)
            if train_mode:
                opt.zero_grad(set_to_none=True)
            pred = model(xb, exists, country if spec.use_country_embedding else None)
            mask = label.bool()
            if mask.sum().item() == 0:
                continue
            loss = ((pred[mask] - yb[mask]) ** 2).mean()
            if train_mode:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                opt.step()
            n = int(mask.sum().item())
            total_loss += float(loss.detach().cpu()) * n
            total_n += n
        return total_loss / max(total_n, 1)

    save_json({"config_hash": config_hash, "model_id": model_id, "refit_id": int(sched_row.refit_id), "feature_cols": feature_cols, "schedule": sched_row.to_dict(), "train_cfg": asdict(train_cfg), "transformer_cfg": asdict(tr_cfg), "spec": asdict(spec)}, paths["metadata"])

    for epoch in range(start_epoch, train_cfg.max_epochs_transformer):
        t0 = time.time()
        train_loss = run_epoch(train_loader, True)
        with torch.no_grad():
            val_loss = run_epoch(val_loader, False)
        improved = val_loss < best_val - 1e-12
        if improved:
            best_val, bad_epochs = val_loss, 0
            torch.save({"epoch": epoch, "model_state": model.state_dict(), "optimizer_state": opt.state_dict(), "best_val": best_val, "bad_epochs": bad_epochs, "config_hash": config_hash}, paths["best"])
        else:
            bad_epochs += 1
        torch.save({"epoch": epoch, "model_state": model.state_dict(), "optimizer_state": opt.state_dict(), "best_val": best_val, "bad_epochs": bad_epochs, "config_hash": config_hash}, paths["latest"])
        print(f"  {model_id} refit {sched_row.refit_id} epoch {epoch:03d}: train={train_loss:.6g}, val={val_loss:.6g}, best={best_val:.6g}, bad={bad_epochs}, {time.time()-t0:.1f}s")
        if bad_epochs >= train_cfg.patience:
            print(f"  Early stopping {model_id} refit {sched_row.refit_id} at epoch {epoch}.")
            break

    if paths["best"].exists():
        ckpt = torch.load(paths["best"], map_location=device)
        model.load_state_dict(ckpt["model_state"])
    model.eval()

    test_ds = MonthPanelDataset(panel["X"], panel["Y"], panel["EXISTS"], panel["LABEL"], panel["COUNTRY"], test_idx)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)
    rows = []
    with torch.no_grad():
        for local_t, (xb, yb, exists, label, country) in enumerate(test_loader):
            month_panel_idx = test_idx[local_t]
            month = pd.Timestamp(months[month_panel_idx])
            xb, exists, country = xb.to(device), exists.to(device), country.to(device)
            pred = model(xb, exists, country if spec.use_country_embedding else None).detach().cpu().numpy().reshape(-1)
            y = yb.numpy().reshape(-1)
            label_np = label.numpy().reshape(-1).astype(bool)
            exists_np = exists.detach().cpu().numpy().reshape(-1).astype(bool)
            use = np.where(label_np & exists_np)[0]
            for j in use:
                rows.append({
                    "eom": month,
                    "id": panel["ID_PANEL"][month_panel_idx, j],
                    "country": panel["COUNTRY_STR_PANEL"][month_panel_idx, j],
                    "me": panel["ME_PANEL"][month_panel_idx, j],
                    "y_true": float(y[j]),
                    "y_pred": float(pred[j]),
                    "model_id": model_id,
                    "refit_id": int(sched_row.refit_id),
                    "test_year": int(sched_row.test_year),
                })
    preds = pd.DataFrame(rows)
    preds.to_parquet(paths["preds"], index=False)
    print(f"  {model_id} refit {sched_row.refit_id}: saved predictions {preds.shape}, R2={oos_r2(preds.y_true, preds.y_pred):.4%}")
    return preds


def train_nn3_refit(model_id, flat_df, feature_cols, sched_row, checkpoint_base, train_cfg, device, config_hash, target_col):
    paths = checkpoint_paths(checkpoint_base, model_id, int(sched_row.refit_id))
    if paths["preds"].exists() and metadata_matches(paths["metadata"], config_hash):
        print(f"  {model_id} refit {sched_row.refit_id}: predictions exist; loading.")
        return pd.read_parquet(paths["preds"])

    train = flat_df[year_mask(flat_df, sched_row.train_start_year, sched_row.train_end_year)].copy()
    val = flat_df[year_mask(flat_df, sched_row.val_start_year, sched_row.val_end_year)].copy()
    test = flat_df[year_mask(flat_df, sched_row.test_start_year, sched_row.test_end_year)].copy()
    if min(len(train), len(val), len(test)) == 0:
        raise ValueError(f"Empty split for {model_id} refit {sched_row.refit_id}")

    x_train = torch.tensor(train[feature_cols].to_numpy(dtype=np.float32))
    y_train = torch.tensor(train[target_col].to_numpy(dtype=np.float32))
    x_val = torch.tensor(val[feature_cols].to_numpy(dtype=np.float32))
    y_val = torch.tensor(val[target_col].to_numpy(dtype=np.float32))
    x_test = torch.tensor(test[feature_cols].to_numpy(dtype=np.float32))

    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=train_cfg.batch_size_nn3, shuffle=True, num_workers=train_cfg.num_workers, drop_last=len(train) > train_cfg.batch_size_nn3)
    val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=train_cfg.batch_size_nn3, shuffle=False, num_workers=train_cfg.num_workers)

    model = GKXStyleNN3(len(feature_cols)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr_nn3, weight_decay=train_cfg.weight_decay)
    start_epoch, best_val, bad_epochs = 0, float("inf"), 0
    if paths["latest"].exists() and metadata_matches(paths["metadata"], config_hash):
        ckpt = torch.load(paths["latest"], map_location=device)
        model.load_state_dict(ckpt["model_state"])
        opt.load_state_dict(ckpt["optimizer_state"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_val = float(ckpt.get("best_val", best_val))
        bad_epochs = int(ckpt.get("bad_epochs", bad_epochs))
        print(f"  {model_id} refit {sched_row.refit_id}: resumed from epoch {start_epoch}.")

    mse = nn.MSELoss(reduction="mean")
    def run_epoch(loader, train_mode):
        model.train(train_mode)
        total_loss, total_n = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if train_mode:
                opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = mse(pred, yb)
            if train_mode and train_cfg.l1_lambda_nn3 > 0:
                loss = loss + train_cfg.l1_lambda_nn3 * l1_linear_weights(model)
            if train_mode:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                opt.step()
            n = len(yb)
            total_loss += float(loss.detach().cpu()) * n
            total_n += n
        return total_loss / max(total_n, 1)

    save_json({"config_hash": config_hash, "model_id": model_id, "refit_id": int(sched_row.refit_id), "feature_cols": feature_cols, "schedule": sched_row.to_dict(), "train_cfg": asdict(train_cfg)}, paths["metadata"])
    for epoch in range(start_epoch, train_cfg.max_epochs_nn3):
        t0 = time.time()
        train_loss = run_epoch(train_loader, True)
        with torch.no_grad():
            val_loss = run_epoch(val_loader, False)
        improved = val_loss < best_val - 1e-12
        if improved:
            best_val, bad_epochs = val_loss, 0
            torch.save({"epoch": epoch, "model_state": model.state_dict(), "optimizer_state": opt.state_dict(), "best_val": best_val, "bad_epochs": bad_epochs, "config_hash": config_hash}, paths["best"])
        else:
            bad_epochs += 1
        torch.save({"epoch": epoch, "model_state": model.state_dict(), "optimizer_state": opt.state_dict(), "best_val": best_val, "bad_epochs": bad_epochs, "config_hash": config_hash}, paths["latest"])
        print(f"  {model_id} refit {sched_row.refit_id} epoch {epoch:03d}: train={train_loss:.6g}, val={val_loss:.6g}, best={best_val:.6g}, bad={bad_epochs}, {time.time()-t0:.1f}s")
        if bad_epochs >= train_cfg.patience:
            print(f"  Early stopping {model_id} refit {sched_row.refit_id} at epoch {epoch}.")
            break

    if paths["best"].exists():
        ckpt = torch.load(paths["best"], map_location=device)
        model.load_state_dict(ckpt["model_state"])
    model.eval()
    preds_all = []
    with torch.no_grad():
        for start in range(0, len(x_test), train_cfg.batch_size_nn3):
            pred = model(x_test[start:start+train_cfg.batch_size_nn3].to(device)).detach().cpu().numpy()
            preds_all.append(pred)
    y_pred = np.concatenate(preds_all)
    preds = pd.DataFrame({
        "eom": test["eom"].to_numpy(),
        "id": test["id"].astype(str).to_numpy(),
        "country": test["excntry"].astype(str).to_numpy(),
        "me": pd.to_numeric(test["me"], errors="coerce").to_numpy(dtype=np.float64) if "me" in test.columns else np.nan,
        "y_true": test[target_col].to_numpy(dtype=np.float64),
        "y_pred": y_pred.astype(np.float64),
        "model_id": model_id,
        "refit_id": int(sched_row.refit_id),
        "test_year": int(sched_row.test_year),
    })
    preds.to_parquet(paths["preds"], index=False)
    print(f"  {model_id} refit {sched_row.refit_id}: saved predictions {preds.shape}, R2={oos_r2(preds.y_true, preds.y_pred):.4%}")
    return preds

## 10. Backtests

In [12]:
# =============================================================================
# 10. Backtests
# =============================================================================

def equal_weight_decile_backtest(preds: pd.DataFrame, n_groups: int = 10) -> pd.DataFrame:
    df = preds.dropna(subset=["eom", "id", "y_true", "y_pred"]).copy()
    df["eom"] = pd.to_datetime(df["eom"])
    rows = []
    for date, g in df.groupby("eom"):
        if len(g) < n_groups:
            continue
        g = g.copy()
        g["decile"] = pd.qcut(g["y_pred"].rank(method="first"), q=n_groups, labels=False) + 1
        dec_ret = g.groupby("decile")["y_true"].mean()
        row = {"eom": date, "n_stocks": len(g)}
        for d in range(1, n_groups + 1):
            row[f"D{d}"] = float(dec_ret.get(d, np.nan))
        row["long_ret"] = row[f"D{n_groups}"]
        row["short_ret"] = row["D1"]
        row["long_short_ret"] = row["long_ret"] - row["short_ret"]
        rows.append(row)
    return pd.DataFrame(rows).sort_values("eom").reset_index(drop=True)


def country_neutral_decile_backtest(preds: pd.DataFrame, n_groups: int = 10) -> pd.DataFrame:
    df = preds.dropna(subset=["eom", "country", "id", "y_true", "y_pred"]).copy()
    df["eom"] = pd.to_datetime(df["eom"])
    rows = []
    for date, gm in df.groupby("eom"):
        ls_by_country = []
        n_total = 0
        for cty, g in gm.groupby("country"):
            if len(g) < n_groups:
                continue
            g = g.copy()
            g["decile"] = pd.qcut(g["y_pred"].rank(method="first"), q=n_groups, labels=False) + 1
            long_ret = g.loc[g["decile"] == n_groups, "y_true"].mean()
            short_ret = g.loc[g["decile"] == 1, "y_true"].mean()
            if np.isfinite(long_ret) and np.isfinite(short_ret):
                ls_by_country.append(long_ret - short_ret)
                n_total += len(g)
        if ls_by_country:
            rows.append({"eom": date, "long_short_ret": float(np.mean(ls_by_country)), "n_countries": len(ls_by_country), "n_stocks": n_total})
    return pd.DataFrame(rows).sort_values("eom").reset_index(drop=True)


def performance_stats(returns: pd.Series, periods_per_year: int = 12) -> dict:
    r = returns.dropna().astype(float)
    if len(r) == 0:
        return {}
    mean_m = float(r.mean())
    vol_m = float(r.std(ddof=1))
    ann_ret = periods_per_year * mean_m
    ann_vol = math.sqrt(periods_per_year) * vol_m if vol_m > 0 else np.nan
    sharpe = ann_ret / ann_vol if ann_vol and ann_vol > 0 else np.nan
    t_stat = mean_m / (vol_m / math.sqrt(len(r))) if vol_m > 0 else np.nan
    cum = (1.0 + r).cumprod()
    dd = cum / cum.cummax() - 1.0
    return {"mean_monthly_return": mean_m, "annualized_return": ann_ret, "annualized_vol": ann_vol, "annualized_sharpe": sharpe, "t_stat": float(t_stat), "max_drawdown": float(dd.min()), "hit_rate": float((r > 0).mean()), "n_months": int(len(r))}

## 11. Main orchestration

In [13]:
# =============================================================================
# 11. Main orchestration
# =============================================================================

def build_model_config_hash(spec, feature_cols, sched_row, window_cfg, train_cfg, tr_cfg, lasso_cfg, lgbm_cfg, target_col, dev_size_cap):
    payload = {
        "script_version": "stage12_clean_15yr_v1",
        "model_spec": asdict(spec),
        "feature_cols": feature_cols,
        "schedule": sched_row.to_dict(),
        "window_cfg": asdict(window_cfg),
        "train_cfg": asdict(train_cfg),
        "transformer_cfg": asdict(tr_cfg),
        "lasso_cfg": asdict(lasso_cfg),
        "target_col": target_col,
        "normalization": "USA:month_rank; before_target_drop; fill_missing_zero",
        "no_positional_embeddings": True,
    }
    # Preserve existing checkpoint hashes for non-LightGBM models.
    # LightGBM-specific settings only affect LightGBM checkpoints.
    if spec.family == "lgbm":
        payload["lgbm_cfg"] = asdict(lgbm_cfg)
    return json_hash(payload)


def run_pipeline(args: argparse.Namespace) -> None:
    data_path = Path(args.data_path)
    output_dir = Path(args.output_dir) / args.run_tag
    checkpoint_base = output_dir / "checkpoints"
    pred_dir = output_dir / "predictions"
    metrics_dir = output_dir / "metrics"
    backtest_dir = output_dir / "backtests"
    for p in [checkpoint_base, pred_dir, metrics_dir, backtest_dir]:
        ensure_dir(p)

    device = torch.device("cuda" if torch.cuda.is_available() and not args.cpu else "cpu")
    print(f"Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    window_cfg = WindowConfig(train_years=args.train_years, val_years=args.val_years, first_year=args.start_year, last_year=args.end_year)
    train_cfg = TrainConfig(seed=args.seed, batch_size_transformer=args.batch_size_transformer, batch_size_nn3=args.batch_size_nn3, max_epochs_transformer=args.max_epochs_transformer, max_epochs_nn3=args.max_epochs_nn3, patience=args.patience, lr_transformer=args.lr_transformer, lr_nn3=args.lr_nn3, weight_decay=args.weight_decay, l1_lambda_nn3=args.l1_lambda_nn3)
    tr_cfg = TransformerConfig(d_model=args.d_model, n_heads=args.n_heads, n_layers=args.n_layers, d_ff=args.d_ff, dropout=args.dropout, country_embed_dim=args.country_embed_dim)
    lasso_cfg = LassoConfig(alphas=tuple(args.lasso_alphas), max_iter=args.lasso_max_iter)
    lgbm_cfg = LGBMConfig(n_estimators=args.lgbm_n_estimators, learning_rate=args.lgbm_learning_rate, num_leaves=args.lgbm_num_leaves, max_depth=args.lgbm_max_depth, min_child_samples=args.lgbm_min_child_samples, subsample=args.lgbm_subsample, colsample_bytree=args.lgbm_colsample_bytree, reg_alpha=args.lgbm_reg_alpha, reg_lambda=args.lgbm_reg_lambda, early_stopping_rounds=args.lgbm_early_stopping_rounds, n_jobs=args.lgbm_n_jobs, seed=args.seed)
    set_global_seed(train_cfg.seed)

    selected_model_ids = args.models if args.models else DEFAULT_MODELS
    for m in selected_model_ids:
        if m not in MODEL_SPECS:
            raise ValueError(f"Unknown model {m}. Available: {list(MODEL_SPECS)}")

    raw = load_panel(data_path, args.start_year, args.end_year)
    validate_columns(raw, REQUIRED_RAW_FEATURES, args.target_col)

    normalized_by_universe = {
        "USA": rank_normalize_no_lookahead(raw, REQUIRED_RAW_FEATURES, args.target_col, "USA")
    }
    del raw
    gc.collect()

    # Build the rolling schedule on the USA Core20 panel.
    schedule_source = normalized_by_universe["USA"]
    schedule = make_rolling_schedule(schedule_source, window_cfg)
    schedule.to_csv(output_dir / "rolling_schedule.csv", index=False)
    print("Rolling schedule:")
    print(schedule.head())
    print("...")
    print(schedule.tail())

    comparison_rows = []
    feature_cache: Dict[str, Tuple[pd.DataFrame, List[str]]] = {}
    panel_cache: Dict[str, dict] = {}
    flat_cache: Dict[str, pd.DataFrame] = {}

    for model_id in selected_model_ids:
        spec = MODEL_SPECS[model_id]
        print("\n" + "=" * 100)
        print(f"MODEL: {model_id}")
        print("=" * 100)
        pooled_path = pred_dir / f"{model_id}_all_oos_predictions.parquet"
        if pooled_path.exists():
            print(f"{model_id}: pooled OOS predictions exist; loading and refreshing metrics/backtests only.")
            pooled = pd.read_parquet(pooled_path)
            r2 = oos_r2(pooled["y_true"].to_numpy(), pooled["y_pred"].to_numpy())
            print(f"\n{model_id} pooled OOS R2: {r2:.6%}; n={len(pooled):,}")

            pooled["year"] = pd.to_datetime(pooled["eom"]).dt.year
            by_year = [{"year": int(yr), "oos_r2": oos_r2(g["y_true"], g["y_pred"]), "n_obs": len(g)} for yr, g in pooled.groupby("year")]
            pd.DataFrame(by_year).to_csv(metrics_dir / f"{model_id}_oos_r2_by_year.csv", index=False)

            bt_global = equal_weight_decile_backtest(pooled)
            bt_global.to_parquet(backtest_dir / f"{model_id}_global_equal_weight_deciles.parquet", index=False)
            stats_global = performance_stats(bt_global["long_short_ret"] if "long_short_ret" in bt_global.columns else pd.Series(dtype=float))
            save_json(stats_global, metrics_dir / f"{model_id}_global_equal_weight_ls_stats.json")

            n_features = 100 if spec.feature_mode == "lag_cloud" else spec.raw_count
            row = {"model_id": model_id, "family": spec.family, "universe": spec.universe, "feature_mode": spec.feature_mode, "n_features": n_features, "pooled_oos_r2": r2, "n_oos_obs": len(pooled)}
            row.update({f"global_ew_ls_{k}": v for k, v in stats_global.items()})
            comparison_rows.append(row)
            comp = pd.DataFrame(comparison_rows).sort_values("pooled_oos_r2", ascending=False)
            comp.to_csv(metrics_dir / "model_comparison_so_far.csv", index=False)
            print("Current comparison:")
            print(comp)
            continue

        norm = normalized_by_universe[spec.universe]
        feature_key = f"{spec.universe}_{spec.feature_mode}_{spec.raw_count}"
        if feature_key not in feature_cache:
            df_features, feature_cols = build_feature_frame_for_model(norm, spec, args.target_col)
            feature_cache[feature_key] = (df_features, feature_cols)
            print(f"Built feature frame {feature_key}: rows={len(df_features):,}, features={len(feature_cols)}")
        else:
            df_features, feature_cols = feature_cache[feature_key]
            print(f"Using cached feature frame {feature_key}: rows={len(df_features):,}, features={len(feature_cols)}")
        save_json({"model_id": model_id, "feature_cols": feature_cols, "n_features": len(feature_cols), "spec": asdict(spec)}, metrics_dir / f"{model_id}_features.json")

        all_refit_preds = []
        if spec.family == "transformer":
            if feature_key not in panel_cache:
                print(f"Panelizing {feature_key}. This may use memory.")
                panel_cache[feature_key] = panelize_for_transformer(df_features, feature_cols, args.target_col)
                print(f"Panel shapes: X={panel_cache[feature_key]['X'].shape}, months={len(panel_cache[feature_key]['months'])}")
            panel = panel_cache[feature_key]
            for _, sched_row in schedule.iterrows():
                cfg_hash = build_model_config_hash(spec, feature_cols, sched_row, window_cfg, train_cfg, tr_cfg, lasso_cfg, lgbm_cfg, args.target_col, args.dev_size_cap)
                preds = train_transformer_refit(model_id, spec, panel, feature_cols, sched_row, checkpoint_base, train_cfg, tr_cfg, device, cfg_hash)
                all_refit_preds.append(preds)
                gc.collect()
                if device.type == "cuda":
                    torch.cuda.empty_cache()
        elif spec.family == "nn3":
            if feature_key not in flat_cache:
                flat_cache[feature_key] = flat_labeled_arrays(df_features, feature_cols, args.target_col)
                print(f"Flat labeled frame {feature_key}: rows={len(flat_cache[feature_key]):,}")
            flat = flat_cache[feature_key]
            for _, sched_row in schedule.iterrows():
                cfg_hash = build_model_config_hash(spec, feature_cols, sched_row, window_cfg, train_cfg, tr_cfg, lasso_cfg, lgbm_cfg, args.target_col, args.dev_size_cap)
                preds = train_nn3_refit(model_id, flat, feature_cols, sched_row, checkpoint_base, train_cfg, device, cfg_hash, args.target_col)
                all_refit_preds.append(preds)
                gc.collect()
                if device.type == "cuda":
                    torch.cuda.empty_cache()
        elif spec.family == "lasso":
            if feature_key not in flat_cache:
                flat_cache[feature_key] = flat_labeled_arrays(df_features, feature_cols, args.target_col)
                print(f"Flat labeled frame {feature_key}: rows={len(flat_cache[feature_key]):,}")
            flat = flat_cache[feature_key]
            for _, sched_row in schedule.iterrows():
                cfg_hash = build_model_config_hash(spec, feature_cols, sched_row, window_cfg, train_cfg, tr_cfg, lasso_cfg, lgbm_cfg, args.target_col, args.dev_size_cap)
                preds = train_lasso_refit(model_id, flat, feature_cols, sched_row, checkpoint_base, lasso_cfg, cfg_hash, args.target_col)
                all_refit_preds.append(preds)
        elif spec.family == "lgbm":
            if feature_key not in flat_cache:
                flat_cache[feature_key] = flat_labeled_arrays(df_features, feature_cols, args.target_col)
                print(f"Flat labeled frame {feature_key}: rows={len(flat_cache[feature_key]):,}")
            flat = flat_cache[feature_key]
            for _, sched_row in schedule.iterrows():
                cfg_hash = build_model_config_hash(spec, feature_cols, sched_row, window_cfg, train_cfg, tr_cfg, lasso_cfg, lgbm_cfg, args.target_col, args.dev_size_cap)
                preds = train_lgbm_refit(model_id, flat, feature_cols, sched_row, checkpoint_base, lgbm_cfg, cfg_hash, args.target_col)
                all_refit_preds.append(preds)
        else:
            raise ValueError(spec.family)

        pooled = pd.concat(all_refit_preds, ignore_index=True)
        pooled_path = pred_dir / f"{model_id}_all_oos_predictions.parquet"
        pooled.to_parquet(pooled_path, index=False)
        r2 = oos_r2(pooled["y_true"].to_numpy(), pooled["y_pred"].to_numpy())
        print(f"\n{model_id} pooled OOS R2: {r2:.6%}; n={len(pooled):,}")

        pooled["year"] = pd.to_datetime(pooled["eom"]).dt.year
        by_year = [{"year": int(yr), "oos_r2": oos_r2(g["y_true"], g["y_pred"]), "n_obs": len(g)} for yr, g in pooled.groupby("year")]
        pd.DataFrame(by_year).to_csv(metrics_dir / f"{model_id}_oos_r2_by_year.csv", index=False)

        bt_global = equal_weight_decile_backtest(pooled)
        bt_global.to_parquet(backtest_dir / f"{model_id}_global_equal_weight_deciles.parquet", index=False)
        stats_global = performance_stats(bt_global["long_short_ret"] if "long_short_ret" in bt_global.columns else pd.Series(dtype=float))
        save_json(stats_global, metrics_dir / f"{model_id}_global_equal_weight_ls_stats.json")

        stats_cn = {}
        if spec.universe == "DEV":
            bt_cn = country_neutral_decile_backtest(pooled)
            bt_cn.to_parquet(backtest_dir / f"{model_id}_country_neutral_deciles.parquet", index=False)
            stats_cn = performance_stats(bt_cn["long_short_ret"] if "long_short_ret" in bt_cn.columns else pd.Series(dtype=float))
            save_json(stats_cn, metrics_dir / f"{model_id}_country_neutral_ls_stats.json")

        row = {"model_id": model_id, "family": spec.family, "universe": spec.universe, "feature_mode": spec.feature_mode, "n_features": len(feature_cols), "pooled_oos_r2": r2, "n_oos_obs": len(pooled)}
        row.update({f"global_ew_ls_{k}": v for k, v in stats_global.items()})
        row.update({f"country_neutral_ls_{k}": v for k, v in stats_cn.items()})
        comparison_rows.append(row)
        comp = pd.DataFrame(comparison_rows).sort_values("pooled_oos_r2", ascending=False)
        comp.to_csv(metrics_dir / "model_comparison_so_far.csv", index=False)
        print("Current comparison:")
        print(comp)

    final = pd.DataFrame(comparison_rows).sort_values("pooled_oos_r2", ascending=False)
    final.to_csv(metrics_dir / "model_comparison_final.csv", index=False)
    print("\nFINAL COMPARISON")
    print(final)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Stage 1+2 clean LASSO/LightGBM/NN3/Transformer rolling-window pipeline")
    parser.add_argument("--data-path", type=str, default="jkp_8dev_100chars_parquet_1980_2024/jkp_8dev_100chars_combined_1980_2024.parquet")
    parser.add_argument("--output-dir", type=str, default="stage12_runs")
    parser.add_argument("--run-tag", type=str, default="stage12_clean_15yr_v1")
    parser.add_argument("--models", nargs="*", default=None, help=f"Subset of models to run. Available: {list(MODEL_SPECS)}")
    parser.add_argument("--target-col", type=str, default="ret_exc_lead1m")
    parser.add_argument("--start-year", type=int, default=1980)
    parser.add_argument("--end-year", type=int, default=2024)
    parser.add_argument("--train-years", type=int, default=15)
    parser.add_argument("--val-years", type=int, default=4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--cpu", action="store_true")
    parser.add_argument("--dev-size-cap", type=int, default=250, help="Max stocks per country-month for developed-country models, ranked by month-t market equity. Use 0 for no cap.")
    parser.add_argument("--batch-size-transformer", type=int, default=2)
    parser.add_argument("--batch-size-nn3", type=int, default=8192)
    parser.add_argument("--max-epochs-transformer", type=int, default=80)
    parser.add_argument("--max-epochs-nn3", type=int, default=80)
    parser.add_argument("--patience", type=int, default=10)
    parser.add_argument("--lr-transformer", type=float, default=1e-3)
    parser.add_argument("--lr-nn3", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--l1-lambda-nn3", type=float, default=1e-5)
    parser.add_argument("--d-model", type=int, default=64)
    parser.add_argument("--n-heads", type=int, default=4)
    parser.add_argument("--n-layers", type=int, default=2)
    parser.add_argument("--d-ff", type=int, default=128)
    parser.add_argument("--dropout", type=float, default=0.10)
    parser.add_argument("--country-embed-dim", type=int, default=4)
    parser.add_argument("--lasso-alphas", nargs="*", type=float, default=[1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2])
    parser.add_argument("--lasso-max-iter", type=int, default=5000)
    parser.add_argument("--lgbm-n-estimators", type=int, default=3000)
    parser.add_argument("--lgbm-learning-rate", type=float, default=0.03)
    parser.add_argument("--lgbm-num-leaves", type=int, default=31)
    parser.add_argument("--lgbm-max-depth", type=int, default=-1)
    parser.add_argument("--lgbm-min-child-samples", type=int, default=200)
    parser.add_argument("--lgbm-subsample", type=float, default=0.80)
    parser.add_argument("--lgbm-colsample-bytree", type=float, default=0.80)
    parser.add_argument("--lgbm-reg-alpha", type=float, default=0.0)
    parser.add_argument("--lgbm-reg-lambda", type=float, default=1.0)
    parser.add_argument("--lgbm-early-stopping-rounds", type=int, default=100)
    parser.add_argument("--lgbm-n-jobs", type=int, default=-1)
    return parser.parse_args()

## Run configuration

Paths are configured for the Google Drive project folder. Output folders are created automatically.


In [14]:
from argparse import Namespace
from pathlib import Path

# ==== GOOGLE DRIVE PROJECT PATHS ====
# My Drive / Colab Notebooks / FDS Project
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/FDS Project")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = str(PROJECT_DIR / "model_runs")
RUN_TAG = "batchA_us_core_15yr_v1"

# Firm-characteristic dataset: US-only parquet.
# USA parquet search locations:
#   1) FDS Project/data/
#   2) FDS Project/
#   3) a subfolder under either location
# Expected filename is usually: USA_100chars_1980_2024.parquet
# This intentionally rejects the combined 8-country parquet so this batch is USA-only.
def find_us_stock_parquet(project_dir=PROJECT_DIR, data_dir=DATA_DIR):
    candidates = []
    for folder in [data_dir, project_dir]:
        if folder.exists():
            candidates.extend(folder.glob("*.parquet"))
            candidates.extend(folder.glob("*/*.parquet"))

    def is_us_file(p: Path) -> bool:
        name = p.name.lower()
        return (
            "usa" in name
            and "100chars" in name
            and "1980_2024" in name
            and "combined" not in name
            and "macro" not in name
        )

    us_candidates = [p for p in candidates if is_us_file(p)]
    if not us_candidates:
        seen = "\n".join(str(p) for p in candidates) if candidates else "No parquet files found."
        raise FileNotFoundError(
            "No US-only stock-characteristic parquet found. Expected a file like "
            "USA_100chars_1980_2024.parquet inside "
            f"{data_dir} or {project_dir}. Parquet files seen:\n{seen}"
        )
    us_candidates = sorted(set(us_candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    print("Using US-only stock data file:", us_candidates[0])
    return str(us_candidates[0])

DATA_PATH = find_us_stock_parquet()
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

RUN_MODELS = [
    'LASSO_CORE20',
    'LGBM_CORE20',
    'NN3_CORE20',
    'TR_CORE20',
    'TR_LAG_CLOUD',
]

args = Namespace(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    run_tag=RUN_TAG,
    models=RUN_MODELS,
    target_col="ret_exc_lead1m",
    start_year=1980,
    end_year=2024,
    train_years=15,
    val_years=4,
    seed=42,
    cpu=False,
    dev_size_cap=0,  # USA-only batch: no developed-country cap needed.
    batch_size_transformer=2,
    batch_size_nn3=8192,
    max_epochs_transformer=80,
    max_epochs_nn3=80,
    patience=10,
    lr_transformer=1e-3,
    lr_nn3=1e-3,
    weight_decay=1e-4,
    l1_lambda_nn3=1e-5,
    d_model=64,
    n_heads=4,
    n_layers=2,
    d_ff=128,
    dropout=0.10,
    country_embed_dim=4,
    lasso_alphas=[1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    lasso_max_iter=5000,
    lgbm_n_estimators=3000,
    lgbm_learning_rate=0.03,
    lgbm_num_leaves=31,
    lgbm_max_depth=-1,
    lgbm_min_child_samples=200,
    lgbm_subsample=0.80,
    lgbm_colsample_bytree=0.80,
    lgbm_reg_alpha=0.0,
    lgbm_reg_lambda=1.0,
    lgbm_early_stopping_rounds=100,
    lgbm_n_jobs=-1,
)

print(args)


Using US-only stock data file: /content/drive/MyDrive/Colab Notebooks/FDS Project/jkp_USA_100chars_1980_2024.parquet
Namespace(data_path='/content/drive/MyDrive/Colab Notebooks/FDS Project/jkp_USA_100chars_1980_2024.parquet', output_dir='/content/drive/MyDrive/Colab Notebooks/FDS Project/model_runs', run_tag='batchA_us_core_15yr_v1', models=['LASSO_CORE20', 'LGBM_CORE20', 'NN3_CORE20', 'TR_CORE20', 'TR_LAG_CLOUD'], target_col='ret_exc_lead1m', start_year=1980, end_year=2024, train_years=15, val_years=4, seed=42, cpu=False, dev_size_cap=0, batch_size_transformer=2, batch_size_nn3=8192, max_epochs_transformer=80, max_epochs_nn3=80, patience=10, lr_transformer=0.001, lr_nn3=0.001, weight_decay=0.0001, l1_lambda_nn3=1e-05, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.1, country_embed_dim=4, lasso_alphas=[1e-05, 3e-05, 0.0001, 0.0003, 0.001, 0.003, 0.01], lasso_max_iter=5000, lgbm_n_estimators=3000, lgbm_learning_rate=0.03, lgbm_num_leaves=31, lgbm_max_depth=-1, lgbm_min_child_sam

## Run pipeline

In [15]:
run_pipeline(args)

Device: cuda
GPU: NVIDIA L4
Loading data from: /content/drive/MyDrive/Colab Notebooks/FDS Project/jkp_USA_100chars_1980_2024.parquet
Raw filtered shape: (3126974, 110)
Date range: 1980-01-31 00:00:00 to 2024-12-31 00:00:00
Countries in file: ['USA']
Rank-normalizing USA by ['eom'] BEFORE dropping missing future returns...
USA normalized shape: (3126974, 31)
Rolling schedule:
   refit_id  test_year  train_start_year  train_end_year  val_start_year  \
0         0       1999              1980            1994            1995   
1         1       2000              1981            1995            1996   
2         2       2001              1982            1996            1997   
3         3       2002              1983            1997            1998   
4         4       2003              1984            1998            1999   

   val_end_year  test_start_year  test_end_year  
0          1998             1999           1999  
1          1999             2000           2000  
2          2000